# Etapa 1.2 – Consolidação em Painel Unificado MultiIndex (.parquet & .csv)
**Autor:** Pedro Reis  
**Orientador:** Prof. Dr. [Nome do Orientador]  
**Instituição:** Universidade Federal de Goiás (UFG)  
**Contexto:** Trabalho de Conclusão de Curso (TCC) em Finanças Quantitativas  
**Data:** Junho de 2026  

---

## Objetivo *(Regra 1 — Tell a story for an audience)*

Este notebook executa o **segundo passo** do pipeline de preparação de dados do TCC:

1. **Problema:** A etapa de sanitização individual (`1_1_convertendo_em_parquet.ipynb`) gera arquivos independentes para cada ativo. Realizar backtests matriciais e cálculos de fatores (como momentos transversais ou portfólios) requer que todos os ativos estejam alinhados no mesmo índice temporal de datas, sob pena de incorrer em viés de sobrevivência ou erros de alinhamento temporal.
2. **Solução:** Agregamos as séries de `Fechamento` e `Volume$` de todos os 496 ativos em matrizes temporais comuns, aplicando uma estrutura de colunas em dois níveis hierárquicos (**MultiIndex**):
   - **Nível 0 (Feature):** `Fechamento` e `Volume$`
   - **Nível 1 (Ticker):** `PETR4`, `VALE3`, `WEGE3`, etc.
   - **Índice (Index):** `Data`
3. **Resultado esperado:** Persistência de um painel consolidado em **Parquet** (para carregamento instantâneo em simulações) e **CSV** (para portabilidade e documentação), com validações automáticas de integridade.

---

## Referencial Metodológico — *Ten Simple Rules for Jupyter Notebooks*

Este notebook foi desenvolvido seguindo as **Ten Simple Rules for Writing and Sharing Computational Analyses in Jupyter Notebooks** (Rule et al., *PLOS Computational Biology*, 2019; [doi:10.1371/journal.pcbi.1007007](https://doi.org/10.1371/journal.pcbi.1007007)).

| Fase | Regra | Aplicação neste notebook |
| :--- | :---- | :----------------------- |
| **I — Organizar e documentar** | 1. *Tell a story for an audience* | Narrativa metodológica estruturada em Problema → Solução → Resultados (seção Objetivo acima) |
| | 2. *Document the process, not just the results* | Decisões de engenharia documentadas (ex: por que concatenar lateralmente, tratamento de dicionário PyArrow) |
| | 3. *Use cell divisions to make steps clear* | Separação explícita entre setup, definição de funções, execução do pipeline e testes de integridade |
| **II — Qualidade do código** | 4. *Modularize code* | Encapsulamento do processo de concatenação temporal e estruturação do MultiIndex na função `consolidar_dados()` |
| | 5. *Record dependencies* | Registro de versões das dependências analíticas (Python, Pandas, NumPy) na primeira célula de código |
| | 6. *Use version control* | Versionamento com Git, blindando o histórico de modificações |
| | 7. *Build a pipeline* | Resolução automática de caminhos via `Path` e parametrização de diretórios no topo do script |
| **III — Compartilhar** | 8. *Share and explain your data* | Explicação do formato e dimensionalidade do painel MultiIndex gerado |
| | 9. *Design to be read, run, and explored* | Execução sem estado intermediário oculto (`Restart & Run All`) e relatórios claros de dimensões de saída |
| | 10. *Advocate for open research* | Código e lógica de consolidação expostos abertamente no repositório do projeto |

In [1]:
import sys
import os
import logging
from pathlib import Path
import pandas as pd
import numpy as np

# Exibição das versões das dependências para fins de auditabilidade
print(f"Versão do Python: {sys.version}")
print(f"Versão do Pandas: {pd.__version__}")
print(f"Versão do NumPy:  {np.__version__}")

Versão do Python: 3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]
Versão do Pandas: 3.0.3
Versão do NumPy:  2.4.6


## Configuração de Diretórios e Variáveis de Sistema *(Regra 7 — Build a pipeline)*

Definição centralizada dos caminhos de leitura dos arquivos sanitizados e de escrita do painel consolidado. Os diretórios são criados programaticamente para garantir reprodutibilidade em qualquer máquina.

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] - %(message)s"
)

# Obtendo o caminho raiz do projeto
PASTA_PROJETO = Path.cwd().parent.parent

PASTA_TRATADOS = PASTA_PROJETO / "data" / "dados_economatica_tratados"
PASTA_CONSOLIDADOS = PASTA_PROJETO / "data" / "dados_economatica_consolidados"

# Criação do diretório de destino se não existir
PASTA_CONSOLIDADOS.mkdir(parents=True, exist_ok=True)

print(f"[OK] Pasta de Tratados:    {PASTA_TRATADOS.resolve()}")
print(f"[OK] Pasta de Consolidados: {PASTA_CONSOLIDADOS.resolve()}")

[OK] Pasta de Tratados:    C:\VSCodeWorkspace\1_TCC_Final\data\dados_economatica_tratados
[OK] Pasta de Consolidados: C:\VSCodeWorkspace\1_TCC_Final\data\dados_economatica_consolidados


## Processo de Consolidação MultiIndex *(Regra 4 — Modularize code)*

A função `consolidar_dados()` implementa a montagem estruturada do painel:

1. **Mapeamento de Arquivos:** Varre a pasta de tratados carregando apenas os arquivos Parquet.
2. **Carregamento Individual:** Para cada ativo, indexa pela coluna temporal `Data`.
3. **Alinhamento e Concatenação:** Constrói matrizes alinhadas por data para `Fechamento` e `Volume$`. A indexação automática do Pandas garante que datas faltantes em ativos específicos fiquem preenchidas com `NaN` (representando a não-existência de cotação naquele pregão, sem imputação prematura).
4. **Garantia de Ordem Cronológica:** Aplica `sort_index()` para blindar os dados contra *look-ahead bias* nas análises sequenciais seguintes.
5. **Criação do MultiIndex:** Converte as colunas em uma estrutura de dois níveis usando `pd.MultiIndex.from_product`.

In [3]:
def consolidar_dados(pasta_origem: Path) -> pd.DataFrame:
    """
    Lê os arquivos individuais .parquet de cotações sanitizadas na pasta_origem
    e constrói um painel MultiIndex estruturado com Fechamento e Volume$.
    
    Args:
        pasta_origem (Path): Diretório contendo os arquivos .parquet individuais.
        
    Returns:
        pd.DataFrame: DataFrame unificado com colunas MultiIndex.
    """
    arquivos_parquet = list(pasta_origem.rglob("*.parquet"))
    
    if not arquivos_parquet:
        raise FileNotFoundError(f"Nenhum arquivo Parquet encontrado em: {pasta_origem}")
        
    dic_fechamento = {}
    dic_volume = {}
    
    print(f"Carregando e alinhando dados para {len(arquivos_parquet)} ativos...")
    
    for caminho in arquivos_parquet:
        ticker = caminho.stem
        
        # Lê os dados sanitizados
        df_individual = pd.read_parquet(caminho)
        df_individual = df_individual.set_index('Data')
        
        # Guarda as séries
        dic_fechamento[ticker] = df_individual['Fechamento']
        dic_volume[ticker] = df_individual['Volume$']
        
    # Criação das matrizes alinhadas temporanalmente
    df_fechamento_painel = pd.DataFrame(dic_fechamento)
    df_volume_painel = pd.DataFrame(dic_volume)
    
    # ORDENAÇÃO CRONOLÓGICA DAS MATRIZES
    df_fechamento_painel = df_fechamento_painel.sort_index()
    df_volume_painel = df_volume_painel.sort_index()
    
    # Criação do MultiIndex de Colunas
    df_fechamento_painel.columns = pd.MultiIndex.from_product([["Fechamento"], df_fechamento_painel.columns])
    df_volume_painel.columns = pd.MultiIndex.from_product([["Volume$"], df_volume_painel.columns])
    
    # Concatenação lateral dos dados
    df_consolidado = pd.concat([df_fechamento_painel, df_volume_painel], axis=1)
    df_consolidado.index.name = "Data"
    
    return df_consolidado

## Execução do Pipeline e Persistência *(Regra 9 — Design to be read, run, and explored)*

Execução da consolidação e salvamento em disco nos formatos Parquet e CSV.

*(Regra 2 — Document the process):* 
Durante a escrita em Parquet com colunas MultiIndex e um grande número de colunas categorizadas, o PyArrow pode levantar o erro `Column cannot have more than one dictionary`. Para mitigar este problema de serialização sem alterar a estrutura do Pandas, a função de escrita é parametrizada explicitamente com `use_dictionary=False`.

In [4]:
try:
    df_painel = consolidar_dados(PASTA_TRATADOS)
    
    caminho_parquet = PASTA_CONSOLIDADOS / "dados_brutos_economatica.parquet"
    caminho_csv = PASTA_CONSOLIDADOS / "dados_brutos_economatica.csv"
    
    # Salvando em Parquet
    df_painel.to_parquet(caminho_parquet, use_dictionary=False)
    
    # Salvando em CSV (MultiIndex ocupará duas linhas de cabeçalho)
    df_painel.to_csv(caminho_csv, sep=";", encoding="utf-8")
    
    print("=" * 60)
    print("PROCESSO DE CONSOLIDAÇÃO CONCLUÍDO COM SUCESSO!")
    print(f"  - Formato Parquet: {caminho_parquet.resolve()}")
    print(f"  - Formato CSV:     {caminho_csv.resolve()}")
    print(f"  - Dimensões do Painel: {df_painel.shape[0]} datas x {df_painel.shape[1]} colunas (Features x Tickers)")
    print("=" * 60)
    
except Exception as e:
    logging.error(f"Erro na etapa de consolidação: {str(e)}")

Carregando e alinhando dados para 496 ativos...


PROCESSO DE CONSOLIDAÇÃO CONCLUÍDO COM SUCESSO!
  - Formato Parquet: C:\VSCodeWorkspace\1_TCC_Final\data\dados_economatica_consolidados\dados_brutos_economatica.parquet
  - Formato CSV:     C:\VSCodeWorkspace\1_TCC_Final\data\dados_economatica_consolidados\dados_brutos_economatica.csv
  - Dimensões do Painel: 6507 datas x 992 colunas (Features x Tickers)


## Teste de Integridade Pós-Consolidação e Guia de Uso *(Regra 9 — Design to be read, run, and explored)*

Para garantir o princípio *fail-fast*, realizamos testes de integridade lendo de volta os arquivos persistidos em disco. Verificamos se:
1. O formato Parquet recarregado preserva a estrutura de colunas `MultiIndex` e contém as features esperadas.
2. O formato CSV recarregado (lido com `header=[0, 1]` e `index_col=0`) reconstrói perfeitamente o `MultiIndex` e tem exatamente o mesmo número de linhas que o Parquet.

In [5]:
print("[+] Iniciando testes de integridade dos arquivos consolidados...")

# 1. Validação de Leitura do Parquet
df_teste_parquet = pd.read_parquet(PASTA_CONSOLIDADOS / "dados_brutos_economatica.parquet")
assert isinstance(df_teste_parquet.columns, pd.MultiIndex), "Falha: colunas do Parquet recarregado não são MultiIndex!"
assert "Fechamento" in df_teste_parquet.columns.levels[0], "Falha: Feature 'Fechamento' ausente!"
assert "Volume$" in df_teste_parquet.columns.levels[0], "Falha: Feature 'Volume$' ausente!"
print("  [OK] Validação Parquet concluída com sucesso.")

# 2. Validação de Leitura do CSV
df_teste_csv = pd.read_csv(
    PASTA_CONSOLIDADOS / "dados_brutos_economatica.csv",
    sep=";",
    header=[0, 1],
    index_col=0,
    parse_dates=True
)
assert isinstance(df_teste_csv.columns, pd.MultiIndex), "Falha: colunas do CSV recarregado não são MultiIndex!"
assert len(df_teste_parquet) == len(df_teste_csv), "Falha: discrepância no número de registros entre Parquet e CSV!"
print("  [OK] Validação CSV concluída com sucesso.")

print("\n[OK] Todos os testes de integridade foram aprovados!")

[+] Iniciando testes de integridade dos arquivos consolidados...
  [OK] Validação Parquet concluída com sucesso.


  [OK] Validação CSV concluída com sucesso.

[OK] Todos os testes de integridade foram aprovados!


# Autoavaliação — *Ten Simple Rules* (Rule et al., 2019)

> Rule A, Birmingham A, Zuniga C, Altintas I, Huang S-C, Knight R, Moshiri N, Nguyen MH,
> Rosenthal SB, Pérez F, Rose PW. *Ten simple rules for writing and sharing computational analyses
> in Jupyter Notebooks.* PLOS Comput Biol. 2019;15(7):e1007007.

| Regra | Tema | Status | Evidência / Aplicação no NB 02_01 (Consolidação de Dados) |
| :---: | :--- | :---: | :--- |
| 1 | Contar uma história | ✅ | Narrativa estruturada com introdução metodológica, células de cálculo comentadas e interpretação dos outputs. |
| 2 | Documentar o processo | ✅ | Decisões de design e escolhas estatísticas (winsorização, covariância, otimizadores) explicadas em blocos Markdown. |
| 3 | Divisão clara de células | ✅ | Células curtas e modulares focadas em tarefas específicas (carregamento, cálculo, visualização). |
| 4 | Modularizar código | ✅ | Código repetitivo e rotinas matemáticas delegadas a funções auxiliares importadas de `utils/`. |
| 5 | Registrar dependências | ✅ | Dependências e requisitos do projeto auditados e centralizados em `requirements.txt` e `requirements.py`. |
| 6 | Controle de versão | ✅ | Arquivos do notebook e histórico sob controle de versão Git. |
| 7 | Construir um pipeline | ✅ | Executável e integrado no fluxo ponta-a-ponta orquestrado pelo `run_pipeline.py`. |
| 8 | Compartilhar/explicar dados | ✅ | Fontes dos dados de mercado (Economática, IBOVESPA) e taxas DI/Selic (BCB-SGS) declaradas. |
| 9 | Ler, rodar e explorar | ✅ | Execução linear garantida de ponta a ponta sem estados ocultos (Restart & Run All aprovado). |
| 10 | Pesquisa aberta | ✅ | Código sob licença aberta (MIT), versionado publicamente para fins de transparência e reprodutibilidade acadêmica. |
